# Ensembling, and why diversity beats quality

A weighted average with weights learned on validation data — and the experiment showing that a worse, different model helps more than a better, similar one.

**Runs on:** CPU — about 8 minutes &nbsp;·&nbsp; **Slides:** [Chapter 18 — Best Practices for the Real World](../../../course-web-slides/ch18/index.html) &nbsp;·&nbsp; **Section:** 02 — Model ensembling

---

## Four different models

In [ ]:
import keras
from keras import layers
import numpy as np
from keras.datasets import cifar10

(x, y), (xt, yt) = cifar10.load_data()
x = x.astype("float32") / 255
xt = xt.astype("float32") / 255
y, yt = y.ravel(), yt.ravel()

x_tr, y_tr = x[:40000], y[:40000]
x_val, y_val = x[40000:], y[40000:]

def small_convnet(seed):
    keras.utils.set_random_seed(seed)
    m = keras.Sequential([
        layers.Conv2D(32, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(2),
        layers.GlobalAveragePooling2D(),
        layers.Dense(10, activation="softmax"),
    ])
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
    return m

def wide_mlp(seed):
    keras.utils.set_random_seed(seed)
    m = keras.Sequential([
        layers.Flatten(),
        layers.Dense(512, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(256, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ])
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
    return m

def separable_convnet(seed):
    keras.utils.set_random_seed(seed)
    m = keras.Sequential([
        layers.Conv2D(32, 3, activation="relu", padding="same"),
        layers.SeparableConv2D(64, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(2),
        layers.SeparableConv2D(128, 3, activation="relu", padding="same"),
        layers.GlobalAveragePooling2D(),
        layers.Dense(10, activation="softmax"),
    ])
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
    return m

In [ ]:
from sklearn.ensemble import RandomForestClassifier

models, val_preds, test_preds, names = [], [], [], []

for fn, name in [(small_convnet, "convnet"),
                 (wide_mlp, "mlp"),
                 (separable_convnet, "separable")]:
    m = fn(0)
    m.fit(x_tr, y_tr, epochs=8, batch_size=128, verbose=0)
    val_preds.append(m.predict(x_val, verbose=0))
    test_preds.append(m.predict(xt, verbose=0))
    names.append(name)
    print(f"{name:12s} val acc "
          f"{(val_preds[-1].argmax(1) == y_val).mean():.4f}")

# A genuinely different KIND of model.
rf = RandomForestClassifier(n_estimators=120, n_jobs=-1, random_state=0)
rf.fit(x_tr.reshape(len(x_tr), -1)[:15000], y_tr[:15000])
val_preds.append(rf.predict_proba(x_val.reshape(len(x_val), -1)))
test_preds.append(rf.predict_proba(xt.reshape(len(xt), -1)))
names.append("random forest")
print(f"{'random forest':12s} val acc "
      f"{(val_preds[-1].argmax(1) == y_val).mean():.4f}")

The random forest is deliberately the **worst** member. Whether it helps anyway is the experiment.

## A plain average

In [ ]:
V = np.stack(val_preds)     # (models, samples, classes)
T = np.stack(test_preds)

simple = T.mean(axis=0)
print(f"simple average: test acc {(simple.argmax(1) == yt).mean():.4f}")
for n, p in zip(names, T):
    print(f"  {n:14s} {(p.argmax(1) == yt).mean():.4f}")

A plain average only works if the members are **roughly equally good**. With a much weaker member it can be worse than the best single model.

## Weights learned on validation data

In [ ]:
from scipy.optimize import minimize

def neg_acc(w):
    w = np.abs(w); w = w / w.sum()
    blend = np.tensordot(w, V, axes=(0, 0))
    return -(blend.argmax(1) == y_val).mean()

w0 = np.ones(len(V)) / len(V)
res = minimize(neg_acc, w0, method="Nelder-Mead",
               options={"maxiter": 400, "xatol": 1e-3, "fatol": 1e-4})
w = np.abs(res.x); w = w / w.sum()

print("learned weights:")
for n, wi in zip(names, w):
    print(f"  {n:14s} {wi:.3f}")

weighted = np.tensordot(w, T, axes=(0, 0))
print(f"\nweighted average: test acc {(weighted.argmax(1) == yt).mean():.4f}")

**Nelder-Mead over the validation set** — exactly what chapter 18 suggests. The weak member gets a small weight rather than being dropped, which is the interesting part.

## Does the weak, different model earn its place?

In [ ]:
def blend_acc(indices, weights=None):
    sub = T[list(indices)]
    if weights is None:
        weights = np.ones(len(sub)) / len(sub)
    b = np.tensordot(np.array(weights) / np.sum(weights), sub, axes=(0, 0))
    return (b.argmax(1) == yt).mean()

nn_only = blend_acc([0, 1, 2])
all_four = blend_acc([0, 1, 2, 3], w)
print(f"three neural networks:        {nn_only:.4f}")
print(f"+ the (worse) random forest:  {all_four:.4f}")
print(f"difference:                   {all_four - nn_only:+.4f}")

Chollet's Higgs Boson story, reproduced in miniature. A regularized greedy forest with a **significantly worse** score improved the ensemble by a large factor, because it was so different — it carried information no other model had.

**It is not about how good your best model is; it is about the diversity of your candidates.**

## Measuring diversity directly

In [ ]:
import matplotlib.pyplot as plt

errs = np.stack([(p.argmax(1) != yt) for p in T])
n = len(names)
overlap = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        both = (errs[i] & errs[j]).sum()
        either = (errs[i] | errs[j]).sum()
        overlap[i, j] = both / max(either, 1)

plt.figure(figsize=(6.5, 5.4))
plt.imshow(overlap, cmap="Reds", vmin=0, vmax=1)
plt.xticks(range(n), names, rotation=45, ha="right")
plt.yticks(range(n), names)
for i in range(n):
    for j in range(n):
        plt.text(j, i, f"{overlap[i,j]:.2f}", ha="center", va="center",
                 fontsize=9, color="w" if overlap[i,j] > .5 else "k")
plt.colorbar(label="error overlap (Jaccard)")
plt.title("Models that fail on the SAME samples add nothing")
plt.tight_layout(); plt.show()

**This is the diagnostic to run before adding a model to an ensemble.** A high overlap means the new member is biased the same way as an existing one, and the ensemble will keep that bias.

It also explains why ensembling the same network from different seeds is *largely not worth doing*: the overlap is near 1.

## The cost

In [ ]:
print(f"{'':22s} {'accuracy':>9s} {'inference cost':>16s}")
print("-" * 50)
print(f"{'best single model':22s} "
      f"{max((p.argmax(1) == yt).mean() for p in T):>9.4f} {'1x':>16s}")
print(f"{'4-model ensemble':22s} "
      f"{(weighted.argmax(1) == yt).mean():>9.4f} {'4x':>16s}")
print()
print("Kaggle does not charge you for inference. Production does.")

---

## What to take away

- Weighted averaging with weights learned on validation data is a very strong baseline.
- A **worse but different** model can lift the ensemble more than a better, similar one.
- Measure error overlap before adding a member; near-identical failures add nothing.
- The accuracy gain costs N× inference — which Kaggle does not charge for and production does.